# Option 5: Policy Learning
**Replication**: Kitagawa & Tetenov (2018) - Who Should Be Treated?
**Data**: NSW Job Training Program (LaLonde, 1986; Dehejia & Wahba, 1999)

**Key Question**: How to optimally assign job training given a budget?

## 1. Setup and Data Loading

In [ ]:
# Download data files if not already present
import os
import urllib.request

BASE_URL = "https://raw.githubusercontent.com/JasmineHao/JasmineHao.github.io/main/econ6083/final-project/notebooks/data/"
DATA_FILES = ['nsw_mixtape.csv']

os.makedirs('data', exist_ok=True)
for fname in DATA_FILES:
    if not os.path.exists(f'data/{fname}'):
        print(f"Downloading {fname} ...")
        urllib.request.urlretrieve(BASE_URL + fname, f'data/{fname}')
        print(f"  Saved to data/{fname}")
    else:
        print(f"Found local: data/{fname}")


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LogisticRegression

# Download REAL NSW data from local file
# Dehejia & Wahba (1999) subset used in many ML applications
df = pd.read_csv('data/nsw_mixtape.csv')

print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nTreatment rate: {df['treat'].mean():.3f}")
df.head()

## 2. Data Exploration

In [ ]:
# Key variables
# re78: Real earnings in 1978 (outcome)
# treat: Treatment assignment (1 = NSW job training)
# covariates: age, educ, black, hisp, marr, nodegree, re74, re75

print("Outcome: Real Earnings 1978")
print(df['re78'].describe())

print("\nTreatment Assignment")
print(df['treat'].value_counts())

print("\nATE (Simple Difference in Means):")
ate_simple = df[df['treat']==1]['re78'].mean() - df[df['treat']==0]['re78'].mean()
print(f"ATE = ${ate_simple:.2f}")

print("\nCovariate balance:")
covs = ['age', 'educ', 'black', 'hisp', 'marr', 'nodegree', 're74', 're75']
print(df.groupby('treat')[covs].mean().round(2))

## 3. Policy Learning Implementation

In [ ]:
def doubly_robust_scores(Y, D, X, n_splits=5, random_state=42):
    """
    Compute doubly robust scores with cross-fitting:
    
    Γ = μ₁(X) - μ₀(X) + D(Y - μ₁(X))/e(X) - (1-D)(Y - μ₀(X))/(1-e(X))
    
    Returns: DR scores (array of CATE estimates for each unit)
    """
    n = len(Y)
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    
    mu0 = np.zeros(n)
    mu1 = np.zeros(n)
    ps = np.zeros(n)
    
    for train_idx, test_idx in kf.split(X):
        X_train, X_test = X[train_idx], X[test_idx]
        Y_train, Y_test = Y[train_idx], Y[test_idx]
        D_train, D_test = D[train_idx], D[test_idx]
        
        # Outcome models
        if (D_train==0).sum() > 10:
            rf0 = RandomForestRegressor(n_estimators=50, random_state=42)
            rf0.fit(X_train[D_train==0], Y_train[D_train==0])
            mu0[test_idx] = rf0.predict(X_test)
        
        if (D_train==1).sum() > 10:
            rf1 = RandomForestRegressor(n_estimators=50, random_state=42)
            rf1.fit(X_train[D_train==1], Y_train[D_train==1])
            mu1[test_idx] = rf1.predict(X_test)
        
        # Propensity score
        ps_model = LogisticRegression(max_iter=1000).fit(X_train, D_train)
        ps[test_idx] = ps_model.predict_proba(X_test)[:, 1]
    
    # Clip propensity scores
    ps = np.clip(ps, 0.05, 0.95)
    
    # Doubly robust scores
    gamma = mu1 - mu0 + D * (Y - mu1) / ps - (1 - D) * (Y - mu0) / (1 - ps)
    
    return gamma

def learn_policy(X, dr_scores, budget=0.5):
    """
    Learn policy π(X) that maximizes welfare subject to budget constraint
    
    Returns: policy (binary array indicating treatment assignment)
    """
    thresholds = np.percentile(dr_scores, np.linspace(0, 100, 100))
    best_welfare = -np.inf
    best_thresh = 0
    
    for thresh in thresholds:
        policy = (dr_scores >= thresh).astype(int)
        if policy.mean() <= budget + 0.02:
            welfare = np.mean(policy * dr_scores)
            if welfare > best_welfare:
                best_welfare = welfare
                best_thresh = thresh
    
    policy = (dr_scores >= best_thresh).astype(int)
    return policy, best_thresh

print("Functions defined successfully!")

## 4. Prepare Data and Run Analysis

In [ ]:
# Define variables
Y = df['re78'].values
D = df['treat'].values

# Covariates
covariates = ['age', 'educ', 'black', 'hisp', 'marr', 'nodegree', 're74', 're75']
X = df[covariates].values

print(f"Outcome shape: {Y.shape}")
print(f"Treatment shape: {D.shape}")
print(f"Covariates shape: {X.shape}")

# Compute DR scores
dr_scores = doubly_robust_scores(Y, D, X)

print("\nDR Score Statistics:")
print(f"Mean: {dr_scores.mean():.2f}")
print(f"Std: {dr_scores.std():.2f}")
print(f"Min: {dr_scores.min():.2f}")
print(f"Max: {dr_scores.max():.2f}")

## 5. Learn Optimal Policy

In [ ]:
# Learn optimal policy with different budgets
budgets = [0.3, 0.5, 0.7]
policy_results = []

for budget in budgets:
    policy, threshold = learn_policy(X, dr_scores, budget=budget)
    
    # Evaluate on observed data
    treated_welfare = Y[policy==1].mean() if policy.sum() > 0 else 0
    
    policy_results.append({
        'budget': budget,
        'treatment_rate': policy.mean(),
        'threshold': threshold,
        'avg_outcome_treated': treated_welfare,
        'n_treated': policy.sum()
    })

policy_df = pd.DataFrame(policy_results)
print("Policy Results by Budget:")
print(policy_df.round(3))

# Compare with naive policies
np.random.seed(42)
random_policy = (np.random.uniform(0, 1, len(Y)) < 0.5).astype(int)

print("\n" + "="*60)
print("Comparison with Alternative Policies:")
print(f"Current RCT assignment: Treat {D.mean():.1%}, Avg outcome ${Y[D==1].mean():.0f}")
print(f"Random (50%): Treat {random_policy.mean():.1%}, Avg outcome ${Y[random_policy==1].mean():.0f}")

opt_policy, opt_thresh = learn_policy(X, dr_scores, budget=0.5)
print(f"Optimal (50%): Treat {opt_policy.mean():.1%}, Avg outcome ${Y[opt_policy==1].mean():.0f}")
print("="*60)

## 6. Analyze Optimal Policy

In [ ]:
# Use 50% budget policy for analysis
policy, threshold = learn_policy(X, dr_scores, budget=0.5)

df['policy'] = policy
df['dr_scores'] = dr_scores

# Compare characteristics of treated vs untreated under optimal policy
print("Characteristics of Treated vs Untreated (Optimal Policy):")
comparison = df.groupby('policy')[covariates + ['re78']].mean()
print(comparison.round(2))

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Policy by education
ax1 = axes[0]
edu_policy = df.groupby('educ')['policy'].mean()
edu_policy.plot(kind='bar', ax=ax1, color='green', alpha=0.7)
ax1.set_xlabel('Education (years)')
ax1.set_ylabel('Treatment Probability')
ax1.set_title('Optimal Policy by Education')
ax1.axhline(y=0.5, color='red', linestyle='--', label='Budget')
ax1.legend()
ax1.grid(alpha=0.3)

# DR score distribution
ax2 = axes[1]
ax2.hist(df[df['policy']==1]['dr_scores'], bins=30, alpha=0.5, label='Treated', color='green')
ax2.hist(df[df['policy']==0]['dr_scores'], bins=30, alpha=0.5, label='Untreated', color='red')
ax2.axvline(x=threshold, color='black', linestyle='--', label=f'Threshold={threshold:.0f}')
ax2.set_xlabel('DR Score (Estimated CATE)')
ax2.set_ylabel('Count')
ax2.set_title('DR Score Distribution by Policy Assignment')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Interpreting Your Results

| Output | What it means | What to look for |
|---|---|---|
| **Simple policy (CATE > 0)** | Treat everyone with positive estimated effect | High share treated, but includes some with small gains |
| **Budget-constrained policy** | Treat top X% by CATE | Better welfare per dollar spent |
| **Policy tree** | Interpretable rule for treatment assignment | Which covariates determine who gets treated? |
| **Welfare comparison** | Average outcome under each policy | Budget-constrained should beat simple if CATEs are heterogeneous |

**Key question**: If you can only treat 30% of the population, whom should you target? The policy tree gives you an interpretable rule.

## Summary

This notebook implements policy learning using REAL NSW data:

1. **Data**: Real job training experiment data (Dehejia & Wahba, 1999)
2. **DR Scores**: Unbiased estimates of CATE for each individual
3. **Optimal Policy**: Treat those with highest DR scores within budget
4. **Targeting**: Policy targets those with higher education, more experience

**Extensions to try**:
- Add fairness constraints (demographic parity)
- Compare different policy classes (linear vs tree)
- Implement dynamic policy learning (multi-period)
- Use Deep Learning as policy class